<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 16. SimCLR — 대조학습 — A Simple Framework for Contrastive Learning of Visual Representations

- **원 논문:** [A Simple Framework for Contrastive Learning of Visual Representations](https://arxiv.org/abs/2002.05709)
- **저자 / 발표:** Ting Chen, Simon Kornblith, Mohammad Norouzi, Geoffrey Hinton · ICML (2020)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** cosine similarity, cross entropy, MLP, 데이터 증강

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

그림 2의 `x → 두 증강 view → encoder f → projection g → NT-Xent` 흐름을 작은 8×8 합성 패턴에 구현합니다. 양성 view의 cosine 유사도가 음성 쌍보다 커지는지, projection 이전 표현 `h`를 고정한 linear probe가 클래스를 분리하는지 확인합니다.

**원 논문과 다른 것**

ImageNet, ResNet-50, 큰 batch와 긴 학습은 생략하고 8×8 합성 영상·작은 MLP·짧은 full-batch 학습을 사용합니다. 따라서 ImageNet 정확도와 표 1~7의 수치를 재현하지 않습니다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2.1 및 그림 2 | 두 개의 확률적 view와 f/g 경로 | 두 view shape와 독립 잡음 |
| §2.1 식 (1) | cosine similarity | 정규화한 표현의 내적 |
| §2.1 식 (2), Algorithm 1 | 2N 표본의 NT-Xent | 대각선 제외·양성 index·유한 gradient |
| §2.2 및 그림 2 | projection 이전 h의 linear evaluation | 고정 encoder의 probe 정확도 |
| §3, 그림 4~5 | 증강과 temperature 미니 실험 | 양성/음성 유사도 차이 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\ell_{i,j}=-\log
\frac{\exp(\operatorname{sim}(z_i,z_j)/\tau)}
{\sum_{k\ne i}\exp(\operatorname{sim}(z_i,z_k)/\tau)}
$$

- $z_i,z_j$는 같은 image의 두 augmentation view, $\tau$는 temperature입니다.
- positive pair는 가깝게, batch의 다른 view는 멀게 만드는 NT-Xent objective입니다.
- augmentation, encoder, projection head, similarity matrix, masked loss를 단계별로 구현합니다.
- $2B$개 view의 embedding $[2B,D]$에서 similarity matrix $[2B,2B]$가 만들어집니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2.1 및 그림 2
- **노트북 구현:** 두 개의 확률적 view와 f/g 경로
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 두 view shape와 독립 잡음를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** ImageNet, ResNet-50, 큰 batch와 긴 학습은 생략하고 8×8 합성 영상·작은 MLP·짧은 full-batch 학습을 사용합니다. 따라서 ImageNet 정확도와 표 1~7의 수치를 재현하지 않습니다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(ACCELERATOR.summary())
print("synthetic-data preprocessing stays on CPU; neural training uses DEVICE")

## 1) 한 샘플에서 두 개의 view 만들기

SimCLR의 label은 사람이 붙이지 않습니다. 같은 원본에서 독립적으로 뽑은 두 view가
양성 쌍이고, batch의 나머지가 음성입니다. 여기서는 crop·color distortion 대신
grayscale 합성 데이터에 맞춘 밝기 jitter와 Gaussian noise를 씁니다.

<details><summary>힌트</summary>
view마다 별도 random draw를 사용하고, 밝기는 sample별 `[N,1,1,1]`로 broadcast하세요.
</details>

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1 및 그림 2
- **이 셀의 책임:** 두 개의 확률적 view와 f/g 경로
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 두 view shape와 독립 잡음. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def make_views(x: Tensor, noise: float = 0.08) -> tuple[Tensor, Tensor]:
    def augment() -> Tensor:
        brightness = 0.8 + 0.4 * torch.rand(x.size(0), 1, 1, 1, device=x.device)
        return (x * brightness + noise * torch.randn_like(x)).clamp(0, 1)

    return augment(), augment()


prototypes = torch.zeros(4, 1, 8, 8)
prototypes[0, :, :, 1:3] = 1
prototypes[1, :, :, 5:7] = 1
prototypes[2, :, 1:3, :] = 1
prototypes[3, :, 5:7, :] = 1
labels = torch.arange(4).repeat_interleave(24)
images = (prototypes[labels] + 0.04 * torch.randn(96, 1, 8, 8)).clamp(0, 1)
view_a, view_b = make_views(images)
assert view_a.shape == view_b.shape == images.shape
assert not torch.equal(view_a, view_b)
print("두 view 평균 절대 차이:", float((view_a - view_b).abs().mean()))

## 2) 식 (2): NT-Xent

`z=[z_i, z_j]`는 크기 `2N`입니다. 자기 자신은 denominator에서 제외하고, 앞쪽 view의
양성 index는 `i+N`, 뒤쪽은 `i-N`입니다. temperature `τ`는 logit의 선명도를 조절합니다.

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2.1 식 (1) / §2.1 식 (2), Algorithm 1
- **이 셀의 책임:** cosine similarity / 2N 표본의 NT-Xent
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 정규화한 표현의 내적 / 대각선 제외·양성 index·유한 gradient. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def nt_xent(z1: Tensor, z2: Tensor, temperature: float = 0.2) -> Tensor:
    if z1.shape != z2.shape:
        raise ValueError("the two views must have the same shape")
    n = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    logits = z @ z.T / temperature
    logits.fill_diagonal_(float("-inf"))
    targets = (torch.arange(2 * n, device=z.device) + n) % (2 * n)
    return F.cross_entropy(logits, targets)


toy_z = torch.randn(6, 8, requires_grad=True)
toy_loss = nt_xent(toy_z, toy_z + 0.01 * torch.randn_like(toy_z))
toy_loss.backward()
assert toy_loss.ndim == 0 and torch.isfinite(toy_loss)
assert toy_z.grad is not None and torch.isfinite(toy_z.grad).all()
print("toy NT-Xent:", round(float(toy_loss), 4))

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §2.1 및 그림 2
- **이 셀의 책임:** 두 개의 확률적 view와 f/g 경로
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 두 view shape와 독립 잡음. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TinyEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_projection = nn.Linear(64, 32)
        self.output_projection = nn.Linear(32, 16)

    def forward(self, x: Tensor) -> Tensor:
        flattened = x.flatten(start_dim=1)
        hidden = F.relu(self.input_projection(flattened))
        return self.output_projection(hidden)


class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_projection = nn.Linear(16, 16)
        self.output_projection = nn.Linear(16, 8)

    def forward(self, hidden: Tensor) -> Tensor:
        projected = F.relu(self.hidden_projection(hidden))
        return self.output_projection(projected)


class TinySimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TinyEncoder()
        self.projector = ProjectionHead()

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        h = self.encoder(x)
        return h, self.projector(h)


model = TinySimCLR()
h, z = model(images[:5])
assert h.shape == (5, 16) and z.shape == (5, 8)

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §2.1 식 (2), Algorithm 1
- **이 셀의 책임:** 2N 표본의 NT-Xent
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 대각선 제외·양성 index·유한 gradient. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
model = ACCELERATOR.move(TinySimCLR())
images_device, labels_device = ACCELERATOR.move(images, labels)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []


def contrastive_update(model, batch, optimizer, temperature=0.2):
    model.train()
    first_view, second_view = make_views(batch)
    _, first_projection = model(first_view)
    _, second_projection = model(second_view)
    loss = nt_xent(
        first_projection,
        second_projection,
        temperature=temperature,
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu())


for _ in range(100):
    losses.append(
        contrastive_update(
            model,
            images_device,
            optimizer,
        )
    )

with torch.no_grad():
    a, b = make_views(images_device)
    ha, za = model(a)
    hb, zb = model(b)
    positive = float(F.cosine_similarity(za, zb).mean().cpu())
    negative = float(F.cosine_similarity(za, zb.roll(1, 0)).mean().cpu())
assert np.mean(losses[-10:]) < np.mean(losses[:10])
assert positive > negative
print(
    f"loss {losses[0]:.3f} → {losses[-1]:.3f}, "
    f"positive={positive:.3f}, negative={negative:.3f}"
)
plt.figure(figsize=(5, 2.5))
plt.plot(losses)
plt.title("SimCLR mini training")
plt.xlabel("step")
plt.ylabel("NT-Xent")
plt.grid(alpha=0.2)
plt.show()

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.2 및 그림 2
- **이 셀의 책임:** projection 이전 h의 linear evaluation
- **Tensor shape 계약:** two views [B,C,H,W] → projections [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 고정 encoder의 probe 정확도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
for parameter in model.encoder.parameters():
    parameter.requires_grad_(False)
model.zero_grad(set_to_none=True)
with torch.no_grad():
    frozen_h = model.encoder(images_device)
probe = ACCELERATOR.move(nn.Linear(frozen_h.size(1), 4))
probe_opt = torch.optim.Adam(probe.parameters(), lr=0.05)
for _ in range(80):
    probe_loss = F.cross_entropy(probe(frozen_h), labels_device)
    probe_opt.zero_grad()
    probe_loss.backward()
    probe_opt.step()
accuracy = (probe(frozen_h).argmax(1) == labels_device).float().mean().item()
assert all(parameter.grad is None for parameter in model.encoder.parameters())
assert accuracy > 0.90
print(f"frozen-h linear probe accuracy={accuracy:.1%}")

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `ImageNet, ResNet-50, 큰 batch와 긴 학습은 생략하고 8×8 합성 영상·작은 MLP·짧은 full-batch 학습을 사용합니다. 따라서 ImageNet 정확도와 표 1~7의 수치를 재현하지 않습니다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.